In [3]:
!pip install -q \
langchain \
langchain-community \
langchain-groq \
langchain-huggingface \
sentence-transformers \
faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.7 MB/s eta 0:00:00


In [ ]:

import os, time
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.documents import Document

# FAQs
faqs = [
    "Shipping takes 3-5 business days.",
    "Orders can be canceled within 24 hours.",
    "Returns are accepted within 30 days.",
    "Refunds are processed in 5-7 business days.",
    "Free shipping applies to orders over $50.",
    "You can track orders from your account page.",
    "Payments accepted: Visa, Mastercard, PayPal.",
    "Gift cards cannot be refunded.",
    "Accounts can be deleted from settings.",
    "Customer support is available 24/7."
]

# Embeddings + FAISS
docs = [Document(page_content=f) for f in faqs]

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(docs, embeddings)

# Groq LLM
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    groq_api_key=os.getenv("GROQ_API_KEY")
)

print("FAQ Bot Ready! Type 'exit' to quit.\n")

while True:
    query = input("You: ")

    if query.lower() == "exit":
        break

    results = vectorstore.similarity_search_with_score(query, k=2)

    context = "\n".join(
        [doc.page_content for doc, _ in results]
    )

    prompt = f"""
Answer ONLY using the FAQs below.

FAQs:
{context}

Question:
{query}

Provide a concise answer.
"""

    response = llm.invoke(prompt)

    print("\nBot:", response.content)

    print("\nTop Matches:")
    for doc, score in results:
        print(f"- {doc.page_content} | Score: {score:.4f}")

    print()

/tmp/ipykernel_1681/1956528448.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAQ Bot Ready! Type 'exit' to quit.

You: hi

Bot: Hello, how can I assist you today?

Top Matches:
- Customer support is available 24/7. | Score: 1.8108
- Free shipping applies to orders over $50. | Score: 1.8165

You: Customer support is available 24/7

Bot: Yes, customer support is available 24/7.

Top Matches:
- Customer support is available 24/7. | Score: 0.0463
- Orders can be canceled within 24 hours. | Score: 1.1383

You: ok

Bot: There is no further information available for your question.

Top Matches:
- Accounts can be deleted from settings. | Score: 1.6628
- Payments accepted: Visa, Mastercard, PayPal. | Score: 1.6730

You: exit


HuggingFace embeddings and Groq Llama 3.1 were used as equivalent substitutes while preserving the same RAG workflow (Embedding → FAISS → Retrieval → LLM Generation).